# Pipeline de Inteligência Artificial: Ebola L-Polymerase 🧬💻

**Módulo Sênior: Deep Learning com Quantificação de Incerteza (MC Dropout)**

Este notebook foi otimizado para o Google Colab (utilize Runtime GPU). Ele ingere a base curada (Tiers 1, 2, 3), extrai *Morgan Fingerprints* via RDKit, e treina a rede aplicando a função `Confidence-Weighted MSE`.

In [ ]:
!pip install rdkit tensorflow pandas numpy matplotlib

import sys
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem

print(f"TensorFlow Version: {tf.__version__}")

### Importação da Infraestrutura Matemática (Seu Drive)

In [ ]:
# Montar o Google Drive para importar nossos scripts em .py
try:
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.append('/content/drive/MyDrive/PROJETO-ANTIVIRAIS-NANOTECNOLOGIA-IA/2-MACHINE-LEARNING/2d-CODIGOS_PYTHON')
except:
    print("Rodando localmente sem Drive mount.")
    sys.path.append('./2d-CODIGOS_PYTHON')

# Importando nosso "Cérebro" customizado
from loss_functions import CW_MSE_Layer
from uncertainty_quantification import build_mc_dropout_model

print("Módulos sêniores carregados com sucesso!")

### Ingestão de Dados e Transformação Molecular (RDKit)

In [ ]:
def smiles_to_morgan(smiles, radius=2, nBits=1024):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
            return np.array(fp)
    except:
        pass
    return np.zeros((nBits,))

# Mock de dados caso o Drive não esteja montado (Para Teste de Fumaça)
print("Extraindo Fingerprints moleculares...")
dummy_smiles = ["CC1=C(C=C(C=C1)O)C=O", "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"]
X_train = np.array([smiles_to_morgan(s) for s in dummy_smiles])

# Construindo y_train como [Valor_Real_EC50, Peso_Confianca]
# Molécula 1: EC50 1.2 uM (Tier 1 = Peso 1.0)
# Molécula 2: EC50 4.5 uM (Tier 3 = Peso 0.5)
y_train = np.array([[1.2, 1.0], [4.5, 0.5]])

print(f"Shape de X_train (Fingerprints): {X_train.shape}")
print(f"Shape de y_train (Valor, Peso): {y_train.shape}")

### Compilação e Treinamento da Rede Neural (CW-MSE)

In [ ]:
# Construindo a Rede de MC Dropout
model = build_mc_dropout_model(input_dim=1024)

# Compilando com a nossa Inovação Matemática
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss=CW_MSE_Layer())

# Treinamento (Smoke Test)
history = model.fit(X_train, y_train, epochs=5, batch_size=2, verbose=1)
print("Treinamento inicializado perfeitamente!")

### Inferência e Quantificação de Incerteza (MC Dropout Sampler)

In [ ]:
def predict_with_uncertainty(model, x_input, num_samples=100):
    """
    Faz múltiplas inferências (stochastic sampling) ativando o Dropout.
    """
    predictions = []
    for _ in range(num_samples):
        # Como usamos MCDropout customizado, training=True não é necessário aqui, ele sempre dropa
        pred = model(x_input).numpy()
        predictions.append(pred)
    
    predictions = np.array(predictions)
    mean_pred = np.mean(predictions, axis=0)
    variance = np.var(predictions, axis=0)
    std_dev = np.std(predictions, axis=0)
    
    return mean_pred, variance, std_dev

print("Realizando amostragem Monte Carlo para predição de incerteza...")
mean, var, std = predict_with_uncertainty(model, X_train, num_samples=50)
print(f"Predição Média M1: {mean[0][0]:.4f} +/- {std[0][0]:.4f}")
print(f"Predição Média M2: {mean[1][0]:.4f} +/- {std[1][0]:.4f}")